# F1 2026 Race Prediction — Model Analysis Notebook

This notebook walks through the complete pipeline:
1. Data loading via FastF1 API
2. Feature engineering
3. Model training with era weighting
4. Accuracy evaluation
5. Prediction for upcoming races

**Key innovation:** Regulation-era weighting — 2026 data counts 2.5x more than 2023 data to adapt to the biggest F1 regulation change since 2022.

## 1. Setup and imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance

# Project imports
from config import (
    DRIVERS_2026, RESULTS_2026, SPRINT_RESULTS_2026,
    CALENDAR_2026, TEAM_COLOURS, ERA_WEIGHTS, POINTS
)
from utils.data_loader import (
    build_full_dataset, compute_rolling_stats,
    compute_standings, build_2026_features_from_config
)
from models.predictor import (
    train_model, predict_race, prepare_features,
    FEATURE_COLS, TEAM_STRENGTH_2026, DRIVER_EXP
)

print(f'FastF1 version: {fastf1.__version__}')
print(f'Pandas version: {pd.__version__}')
print('All imports OK')

## 2. Load dataset

In [ ]:
# Load full dataset: 2023 + 2024 history + 2026 real data
# First run takes ~3-5 minutes (downloads from FastF1)
# Subsequent runs are instant (cached locally)

print('Loading dataset...')
df = build_full_dataset(max_historical_rounds=10)

print(f'\nDataset summary:')
print(f'  Total rows: {len(df):,}')
print(f'  Seasons: {sorted(df["Year"].unique())}')
print(f'  Drivers: {df["Driver"].nunique()}')
print(f'  Races: {df["Round"].nunique()}')
print(f'  Features: {list(df.columns)}')

In [ ]:
# Show breakdown by season
season_summary = df.groupby('Year').agg(
    Rows=('Driver', 'count'),
    Races=('Round', 'nunique'),
    Drivers=('Driver', 'nunique'),
    AvgFinish=('FinishPos', 'mean'),
    EraWeight=('EraWeight', 'first')
).round(2)

print('Season breakdown:')
print(season_summary.to_string())
print()
print('Note: 2026 era weight is 2.5x -- new regulations make this data most predictive')

## 3. Feature engineering

In [ ]:
# Compute rolling features
rolling_df = compute_rolling_stats(df, window=5)
rolling_df = prepare_features(rolling_df)

print('Rolling features computed:')
feature_cols = ['Driver', 'Year', 'Round', 'GridPosition', 'FinishPos',
                'AvgFinish', 'AvgGrid', 'RecentForm', 'DNFRate',
                'MedianPaceDelta', 'PaceConsistency', 'TeamStrength', 'DriverExp']
available = [c for c in feature_cols if c in rolling_df.columns]
print(rolling_df[available].tail(10).to_string(index=False))

In [ ]:
# Feature correlation heatmap
numeric_features = ['GridPosition', 'AvgFinish', 'AvgGrid', 'RecentForm',
                    'DNFRate', 'MedianPaceDelta', 'PaceConsistency',
                    'TeamStrength', 'DriverExp', 'FinishPos']

available_features = [f for f in numeric_features if f in rolling_df.columns]
corr_matrix = rolling_df[available_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdYlGn_r',
    center=0,
    ax=ax,
    square=True
)
ax.set_title('Feature Correlation Matrix\n(FinishPos = target variable)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/plots/feature_correlation.png')

## 4. Model training with era weighting

In [ ]:
# Train the model
print('Training GradientBoosting + RandomForest ensemble...')
model_obj = train_model(rolling_df)

print(f'\nModel performance:')
print(f'  Cross-validated MAE: {model_obj["mae"]:.3f} finishing positions')
print(f'  Training samples: {model_obj["n_samples"]:,}')
print(f'  Features used: {model_obj["features"]}')

In [ ]:
# Era weighting visualisation
era_data = pd.DataFrame([
    {'Season': '2023', 'Weight': ERA_WEIGHTS[2023], 'Races': 10,
     'Regulation': 'Old regs'},
    {'Season': '2024', 'Weight': ERA_WEIGHTS[2024], 'Races': 10,
     'Regulation': 'Old regs'},
    {'Season': '2026', 'Weight': ERA_WEIGHTS[2026], 'Races': 2,
     'Regulation': 'NEW REGS'},
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colours = ['#888888', '#aaaaaa', '#E8002D']

# Bar chart of weights
axes[0].bar(era_data['Season'], era_data['Weight'], color=colours)
axes[0].set_title('Era Weight per Season\n(sample_weight in sklearn fit)', fontsize=12)
axes[0].set_ylabel('Sample weight multiplier')
for i, (_, row) in enumerate(era_data.iterrows()):
    axes[0].text(i, row['Weight'] + 0.05, f"{row['Weight']}x",
                 ha='center', fontweight='bold')

# Effective training influence
era_data['EffectiveRows'] = era_data['Races'] * 20 * era_data['Weight']
axes[1].bar(era_data['Season'], era_data['EffectiveRows'], color=colours)
axes[1].set_title('Effective Training Influence\n(rows x weight)', fontsize=12)
axes[1].set_ylabel('Effective training contribution')

plt.suptitle('Regulation Era Weighting -- Adapting to 2026 Rule Changes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/era_weighting.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importance_df = model_obj['importance']

fig, ax = plt.subplots(figsize=(10, 6))
colours_imp = ['#E8002D' if imp > 0.15 else '#3671C6' if imp > 0.08 else '#888888'
               for imp in importance_df['Importance']]

bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
               color=colours_imp)
ax.set_xlabel('Feature Importance (GradientBoosting)')
ax.set_title('Feature Importance\nRed = high impact, Blue = medium, Gray = low', fontsize=12)
ax.invert_yaxis()

for bar, val in zip(bars, importance_df['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: GridPosition (qualifying) is the strongest single predictor')

## 5. 2026 championship standings

In [ ]:
# Current standings including sprint points
standings = compute_standings(results_dict=RESULTS_2026)
print('2026 Championship Standings (after R1 Australia + R2 China + China Sprint):')
print(standings[['Name', 'Team', 'Points']].to_string())

In [ ]:
# Standings bar chart
top_drivers = standings.head(12).reset_index()

colours_drv = [TEAM_COLOURS.get(team, '#888888') for team in top_drivers['Team']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(top_drivers['Name'].str.split().str[-1],
              top_drivers['Points'], color=colours_drv)

for bar, pts in zip(bars, top_drivers['Points']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(pts)), ha='center', fontsize=10, fontweight='bold')

ax.set_title('2026 F1 Championship Standings\n(Race + Sprint points, after R2 China)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Championship Points')
ax.set_xlabel('Driver')
plt.xticks(rotation=45, ha='right')

# Add team colour legend
teams_shown = top_drivers['Team'].unique()
patches = [mpatches.Patch(color=TEAM_COLOURS.get(t, '#888'),
           label=t) for t in teams_shown]
ax.legend(handles=patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('../results/plots/championship_standings.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Japan R3 prediction

In [ ]:
# Get latest features per driver
latest_features = (
    rolling_df
    .sort_values(['Driver', 'Year', 'Round'])
    .groupby('Driver').last()
    .reset_index()
)

# Fill drivers missing from training data
known = set(latest_features['Driver'].unique())
filler = []
for drv in DRIVERS_2026:
    if drv not in known:
        info = DRIVERS_2026[drv]
        filler.append({
            'Driver': drv, 'Team': info[1],
            'GridPosition': 15.0, 'FinishPos': 15.0,
            'AvgFinish': 15.0, 'AvgGrid': 15.0,
            'RecentForm': 2.0, 'DNFRate': 0.1,
            'MedianPaceDelta': 0.0, 'PaceConsistency': 1.5,
            'EraWeight': 2.5,
        })
if filler:
    latest_features = pd.concat([latest_features, pd.DataFrame(filler)],
                                  ignore_index=True)

# Predict Japan R3
japan_prediction = predict_race(model_obj, latest_features)

print('Predicted finishing order -- Japan GP (Suzuka, March 29 2026):')
print()
display_cols = ['PredictedPos', 'Driver', 'Name', 'Team', 'PredictedPoints', 'Confidence']
available_display = [c for c in display_cols if c in japan_prediction.columns]
print(japan_prediction[available_display].to_string(index=False))

In [ ]:
# Save prediction to CSV
import os
os.makedirs('../results', exist_ok=True)

# Build predictions CSV
pred_rows = []
for _, row in japan_prediction.iterrows():
    pred_rows.append({
        'Round':          3,
        'Race':           'Japan GP',
        'Circuit':        'Suzuka',
        'Date':           '2026-03-29',
        'Driver':         row['Driver'],
        'Name':           row.get('Name', row['Driver']),
        'Team':           row['Team'],
        'PredictedPos':   int(row['PredictedPos']),
        'PredictedPts':   int(row.get('PredictedPoints', 0)),
        'Confidence':     row.get('Confidence', 'Medium'),
        'ActualPos':      '',   # filled after race
        'ActualPts':      '',   # filled after race
        'Error':          '',   # filled after race
    })

pred_df = pd.DataFrame(pred_rows)

# Append to master predictions file
pred_file = '../results/predictions.csv'
if os.path.exists(pred_file):
    existing = pd.read_csv(pred_file)
    # Remove old R3 predictions if any
    existing = existing[existing['Round'] != 3]
    combined = pd.concat([existing, pred_df], ignore_index=True)
else:
    combined = pred_df

combined.to_csv(pred_file, index=False)
print(f'Predictions saved to results/predictions.csv')
print(f'Total predictions stored: {len(combined)} rows')

In [ ]:
# Predicted podium visualisation
top10 = japan_prediction.head(10)

fig, ax = plt.subplots(figsize=(12, 6))

driver_labels = top10['Name'].str.split().str[-1] if 'Name' in top10.columns \
                else top10['Driver']
bar_colours = [TEAM_COLOURS.get(t, '#888888') for t in top10['Team']]

bars = ax.bar(range(len(top10)), top10['PredictedPoints'],
              color=bar_colours, width=0.7)

ax.set_xticks(range(len(top10)))
ax.set_xticklabels(driver_labels, rotation=45, ha='right')
ax.set_ylabel('Predicted Championship Points')
ax.set_title('Japan GP (R3) -- Predicted Top 10\nSuzuka | March 29, 2026',
             fontsize=13, fontweight='bold')

for i, (bar, row) in enumerate(zip(bars, top10.itertuples())):
    pos_label = f'P{int(row.PredictedPos)}'
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            pos_label, ha='center', fontsize=10, fontweight='bold')

# Team legend
teams_shown = top10['Team'].unique()
patches = [mpatches.Patch(color=TEAM_COLOURS.get(t, '#888'), label=t)
           for t in teams_shown]
ax.legend(handles=patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('../results/plots/japan_r3_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/plots/japan_r3_prediction.png')

## 7. After the race — fill in actual results

Run this cell after Japan R3 (March 29) to compute accuracy.

In [ ]:
# FILL IN AFTER JAPAN RACE
# Replace with actual finishing positions
japan_actual = {
    # 'RUS': 1,
    # 'ANT': 2,
    # etc.
}

if japan_actual:
    pred_dict = dict(zip(japan_prediction['Driver'],
                         japan_prediction['PredictedPos']))

    common = set(japan_actual.keys()) & set(pred_dict.keys())
    errors = [abs(pred_dict[d] - japan_actual[d]) for d in common]
    mae    = sum(errors) / len(errors)

    actual_top3 = {d for d, p in japan_actual.items() if p <= 3}
    pred_top3   = {d for d, p in pred_dict.items()   if p <= 3}
    podium_hits = len(actual_top3 & pred_top3)

    actual_top10 = {d for d, p in japan_actual.items() if p <= 10}
    pred_top10   = {d for d, p in pred_dict.items()   if p <= 10}
    top10_hits   = len(actual_top10 & pred_top10)

    print(f'Japan R3 Prediction Accuracy:')
    print(f'  MAE: {mae:.2f} positions')
    print(f'  Podium drivers predicted correctly: {podium_hits}/3')
    print(f'  Top 10 drivers predicted correctly: {top10_hits}/10')

    # Update predictions CSV with actual results
    pred_df_updated = pd.read_csv('../results/predictions.csv')
    for drv, actual_pos in japan_actual.items():
        mask = (pred_df_updated['Round'] == 3) & (pred_df_updated['Driver'] == drv)
        if mask.any():
            pred_df_updated.loc[mask, 'ActualPos'] = actual_pos
            pred_df_updated.loc[mask, 'ActualPts'] = POINTS.get(actual_pos, 0)
            pred_pos = pred_df_updated.loc[mask, 'PredictedPos'].values[0]
            pred_df_updated.loc[mask, 'Error'] = abs(int(pred_pos) - actual_pos)

    pred_df_updated.to_csv('../results/predictions.csv', index=False)
    print('Results updated in predictions.csv')
else:
    print('Race not yet completed. Fill in japan_actual dict after March 29.')

## 8. Model summary

In [ ]:
print('=' * 55)
print('F1 2026 RACE PREDICTION MODEL -- SUMMARY')
print('=' * 55)
print(f'Training data:')
print(f'  Seasons: 2023, 2024, 2026')
print(f'  Total rows: {len(rolling_df):,}')
print(f'  2026 rows: {len(rolling_df[rolling_df["Year"]==2026]):,}')
print()
print(f'Model:')
print(f'  Architecture: GBM (60%) + RandomForest (40%) ensemble')
print(f'  Cross-validated MAE: {model_obj["mae"]:.3f} positions')
print(f'  Features: {len(model_obj["features"])}')
print()
print(f'Era weighting:')
for year, weight in ERA_WEIGHTS.items():
    print(f'  {year}: {weight}x')
print()
print(f'Japan R3 prediction: {japan_prediction.iloc[0]["Driver"]} wins')
print('=' * 55)